# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [26]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [27]:
X = spaceship.drop(columns=["Transported"])
y = spaceship["Transported"]

In [28]:
# drop PassengerId
X = X.drop(columns=["PassengerId"], errors="ignore")

# define numeric and categorical columns
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns

# pipelines
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# combine
preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols)
])

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((6954, 12), (1739, 12), (6954,), (1739,))

In [37]:
rf_pipe = Pipeline([
    ("preprocess", preprocess),  # your transformer from above
    ("model", RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1))
])

cv = cross_val_score(rf_pipe, X_train, y_train, cv=5, scoring="accuracy", n_jobs=-1)
rf_pipe.fit(X_train, y_train)
test_acc = accuracy_score(y_test, rf_pipe.predict(X_test))
print(f"RF CV mean={cv.mean():.4f} (±{cv.std():.4f}) | Test acc={test_acc:.4f}")


RF CV mean=0.7912 (±0.0037) | Test acc=0.7913


- Evaluate your model

In [ ]:
y_pred = rf_pipe.predict(X_test)

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Confusion Matrix:
[[671 190]
 [173 705]]

Classification Report:
              precision    recall  f1-score   support

       False       0.80      0.78      0.79       861
        True       0.79      0.80      0.80       878

    accuracy                           0.79      1739
   macro avg       0.79      0.79      0.79      1739
weighted avg       0.79      0.79      0.79      1739



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

#### Early grid (small search) — exploratory only, not used in final model
param_grid = {
    "model__n_estimators": [20, 40, 60],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None],
    "model__bootstrap": [True]
}

- Run Grid Search

grid = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    refit=True,      
    return_train_score=False
)

grid.fit(X_train, y_train)

- Evaluate your model

print("Best params:", grid.best_params_)
print(f"Best CV accuracy: {grid.best_score_:.4f}")

best_rf_pipe = grid.best_estimator_
test_acc_tuned = accuracy_score(y_test, best_rf_pipe.predict(X_test))
print(f"Test accuracy (tuned): {test_acc_tuned:.4f}")

After GridSearch, the best hyperparameter set achieved CV = 0.7987 but Test = 0.7849, which is below the baseline (0.7913). We conclude that tuning did not generalize better; we keep the baseline as the final model for this dataset.

In [ ]:
### Final model
### RandomizedSearchCV with the wider search is our final choice.

In [ ]:
param_dist = {
    "model__n_estimators": list(range(200, 701, 50)),   # 200..700 de 50 en 50
    "model__max_depth": [None, 10, 15, 20, 25, 30],
    "model__min_samples_split": list(range(2, 11)),     # 2..10
    "model__min_samples_leaf": [1, 2, 3, 4],
    "model__max_features": ["sqrt", "log2"],            # evita None
    "model__bootstrap": [True]
}

In [ ]:
rand = RandomizedSearchCV(
    estimator=rf_pipe,                
    param_distributions=param_dist,
    n_iter=40,                         
    cv=3,                             
    scoring="accuracy",
    n_jobs=-1,                        
    verbose=1,                        
    random_state=42,
    refit=True
)

In [32]:
## This is where the system can work overnight
if False:  # RandomizedSearchCV (heavy computation - disabled by default)
    rand.fit(X_train, y_train)

In [33]:
## Results and evaluation 

if False:
    # RandomizedSearchCV (heavy computation - disabled by default)
    print("Best params:", rand.best_params_)
print(f"Best CV acc (cv=3): {rand.best_score_:.4f}")

best_rf_pipe = rand.best_estimator_
from sklearn.metrics import accuracy_score
test_acc_tuned = accuracy_score(y_test, best_rf_pipe.predict(X_test))
print(f"Test acc (tuned): {test_acc_tuned:.4f}")

Best CV acc (cv=3): 0.7863
Test acc (tuned): 0.7867


In [36]:
## (Optional, when you have time) Validate only that best model with cv=5 to see stability:

if False:
    # RandomizedSearchCV (heavy computation - disabled by default)
    from sklearn.model_selection import cross_val_score
    cv5 = cross_val_score(best_rf_pipe, X_train, y_train, cv=5, scoring="accuracy")
    print(f"CV=5 mean={cv5.mean():.4f} (±{cv5.std():.4f})")

### Conclusion

After performing RandomizedSearchCV, the best set of hyperparameters for the RandomForestClassifier achieved a cross-validation accuracy of **0.7863** and a test accuracy of **0.7867**.  
These results are very close to each other, which indicates that the tuned model generalizes well and is stable.  
Therefore, we select this tuned RandomForestClassifier as the final model for this dataset.